In [1]:
import pandas as pd

In [2]:
#HIGHDIVERSITY
df1 = pd.read_csv('/Users/francisco/Documents/Scripts/DeNovo_PfHDAC1/LigBuilder_results/Build_PfHDAC1_HIGHDIV_CONCAT/result/process_PfHDAC1/dados_sa_score_filtrado.csv')
df1['Dataset'] = 'CONCAT'
df2 = pd.read_csv('/Users/francisco/Documents/Scripts/DeNovo_PfHDAC1/LigBuilder_results/Build_PfHDAC1_HIGHDIV_CRAFT/result/process_PfHDAC1/dados_sa_score_filtrado.csv')
df2['Dataset'] = 'CRAFT'
df3 = pd.read_csv('/Users/francisco/Documents/Scripts/DeNovo_PfHDAC1/LigBuilder_results/Build_PfHDAC1_HIGHDIV_LANA/result/process_PfHDAC1/dados_sa_score_filtrado.csv')
df3['Dataset'] = 'LANAPDB'
df4 = pd.read_csv('/Users/francisco/Documents/Scripts/DeNovo_PfHDAC1/LigBuilder_results/Build_PfHDAC1_HIGHDIV_MAY/result/process_PfHDAC1/dados_sa_score_filtrado.csv')
df4['Dataset'] = 'MAYBRIDGE'
df5 = pd.read_csv('/Users/francisco/Documents/Scripts/DeNovo_PfHDAC1/LigBuilder_results/Build_PfHDAC1_HIGHDIV_ENAMINE/result/process_PfHDAC1/dados_sa_score_filtrado.csv')
df5['Dataset'] = 'ENAMINE'

In [3]:
#CONCATENATE
df_concat_highdiv = pd.concat([df1, df2, df3, df4, df5 ], ignore_index=True)


In [ ]:
from chembl_structure_pipeline import standardizer
from rdkit import Chem
from rdkit import RDLogger
import logging

# Function to standardize SMILES
def standardize_smiles_stage(df):
    """Standardizes SMILES from the 'SMILES' column."""
    print("Standardizing SMILES...")
    smiles_col = 'SMILES'
    
    # 1. Disable RDKit logs
    RDLogger.DisableLog('rdApp.*')
    
    # 2. Disable chembl_structure_pipeline and root logger temporarily
    logger_chembl = logging.getLogger('chembl_structure_pipeline')
    original_level_chembl = logger_chembl.level
    logger_chembl.setLevel(logging.ERROR)  # Only critical errors
    
    # Also suppress root logger in case the pipeline uses logging.info directly
    logging.getLogger().setLevel(logging.ERROR)
    
    def standardize_smiles(s):
        try:
            mol = Chem.MolFromSmiles(s, sanitize=True)
            if mol is None:
                return None
            mol_block = Chem.MolToMolBlock(mol)
            std_mol_block = standardizer.standardize_molblock(mol_block)
            std_mol = Chem.MolFromMolBlock(std_mol_block)
            return Chem.MolToSmiles(std_mol, kekuleSmiles=True)
        except Exception:
            return None

    # Apply standardization
    temp_col = 'standardized_smiles'
    df[temp_col] = df[smiles_col].apply(standardize_smiles)
    
    # Restore logs (optional)
    RDLogger.EnableLog('rdApp.*')
    logger_chembl.setLevel(original_level_chembl)
    logging.getLogger().setLevel(logging.WARNING)  # Restore root logger to warning
    
    # Remove failures
    initial_count = len(df)
    df_clean = df.dropna(subset=[temp_col]).copy()
    removed = initial_count - len(df_clean)
    
    # Replace original column and clean up
    df_clean[smiles_col] = df_clean[temp_col]
    df_clean = df_clean.drop(columns=[temp_col])
    
    print(f"Removed {removed} compounds during SMILES standardization.")
    return df_clean.reset_index(drop=True)

print("\nProcessing df_concat_highdiv...")
df_concat_highdiv = standardize_smiles_stage(df_concat_highdiv)

[15:12:12] Initializing Normalizer


Processing df_concat_druglike...
Standardizing SMILES...
Removed 0 compounds during SMILES standardization.

Processing df_concat_highdiv...
Standardizing SMILES...
Removed 0 compounds during SMILES standardization.


In [ ]:
df_concat_highdiv.to_csv('/Users/francisco/Documents/Scripts/DeNovo_PfHDAC1/Analysis/concat_datasets_highdiv.csv', index=False)